<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part1_pixels/04_coding_images_and_classic_classifiers/notebooks/04_coding_images_and_classic_classifiers_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 04: Coding Images and Classic Classifiers

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Every automated method in this course is judged against human coding, so this notebook starts
there. You write down what you want to measure in a codebook, code a sample of protest photos
yourself, and check how consistent your own coding is. Only then do you train a classifier, on
the handcrafted features from module 03, and test it against your codes. Its limits are the
reason Part 2 of the course exists.

In this notebook you will:

- choose a unit of analysis and work with a short codebook for protest images
- code 40 images with a small labeling tool inside the notebook, and recode some of them later
- measure agreement with percent agreement, Cohen's kappa and Krippendorff's alpha
- compare labels that come for free (the Commons category an image sits in) with your own codes
- train and cross-validate a logistic regression and a random forest on handcrafted features
- test the classifier against your codes, look at its mistakes, and meet the semantic gap

**Data.** Photos from the Wikimedia Commons categories on demonstrations and protests in the
Netherlands, and King's Day photos as crowds that are not protests (both from module 02). The
course's data helper downloads them for you.

This notebook runs on a normal CPU runtime. Coding the images takes about 20 minutes of your time.

## 0. Setup

`krippendorff` computes Krippendorff's alpha; everything else is preinstalled in Colab. The
labeling tool uses `ipywidgets`, which Colab supports out of the box.

In [ ]:
!pip install -q krippendorff

In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import ipywidgets as widgets
from IPython.display import display
import krippendorff
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}

COURSE_DATA_URL = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/course_data.py"
if not Path("course_data.py").exists():
    response = requests.get(COURSE_DATA_URL, headers=HEADERS, timeout=30)
    response.raise_for_status()  # stop if the download failed, rather than saving an error page
    Path("course_data.py").write_text(response.text)
import course_data
from course_data import load_rgb

def show_grid(paths, labels=None, n_cols=8, size=1.6):
    """Show images in a grid, with an optional label under each."""
    paths = list(paths)
    labels = list(labels) if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        ax.imshow(load_rgb(path, 300))
        ax.set_title(str(label)[:24], fontsize=7)
    plt.tight_layout()
    plt.show()

protests = course_data.get_protest_corpus()
koningsdag = course_data.get_koningsdag_corpus()

## 1. What to code: the unit and the codebook

The first decision is the **unit of analysis**. For images it could be the whole image, a region
(the foreground), each person in the image, or the social media post the image belongs to,
together with its caption. Each choice answers different questions. Here the unit is the
**whole image**.

The second is the **codebook**: for each variable, a definition, examples and rules for the hard
cases. A good visual codebook describes what must be *visible*, not what the coder knows or
guesses from context. Here is a short one with four yes/no variables.

| Variable | Code "yes" when... | Code "no" when... |
|---|---|---|
| **protest** | the image shows people engaged in a protest: marching, gathering with signs or banners, a blockade or sit-in, chanting. A single person holding a protest sign counts. | the image shows only objects (a poster on a wall, an empty square), police without protesters, or a crowd with no visible sign of protest (a festival, a market, a celebration) |
| **crowd** | roughly ten or more people are visible, including partly visible people in the background | fewer than about ten people are visible |
| **police** | police officers, police vehicles or barriers marked as police are visible | no police are visible (security guards and stewards don't count) |
| **signs** | at least one sign, banner, placard or flag with a message or symbol is carried or displayed by people | no signs, or only ordinary signs such as street names, shop signs and traffic signs |

Rules for all variables: code only what you can **see** in the image, not what you know about the
event. If you can't tell, choose the answer that requires less inference (usually "no"), and
note the image if you think the codebook needs a new rule.

> **✏️ Exercise 1**
>
> Before you start coding, think of three images that this codebook would make hard to code, for
> example a Pride parade, a convoy of farmers' tractors with banners but no people visible, or a
> close-up of a protest sign with no hands in the frame. For each, decide how the codebook should
> handle it and write the rule you would add.

*Your answer here*

## 2. Coding 40 images

We draw a sample of 30 protest photos and 10 King's Day photos and shuffle them, so you don't know
which source an image comes from while you code it. Knowing would bias your judgment.

The labeling tool below shows one image at a time with a row of buttons per variable. Choose
"no" or "yes" for each and click **Save and next**. Your codes are saved to `my_codes.csv` after
every image, so you can stop and run the cell again later to continue where you left off.

*If the tool doesn't display in your environment, run
`coding_sample[["title", "local_path"]].to_csv("my_codes_template.csv", index=False)`, fill in
the four columns with 0 and 1 in a spreadsheet, and save the result as `my_codes.csv`.*

In [ ]:
VARIABLES = ["protest", "crowd", "police", "signs"]

coding_sample = course_data.coding_sample(protests, koningsdag)   # 30 protest + 10 King's Day photos, shuffled

def code_images(table, out_csv, variables=VARIABLES):
    """Code images one at a time with yes/no buttons; save every answer to out_csv."""
    out_csv = Path(out_csv)
    done = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame(columns=["title", *variables])
    todo = table[~table["title"].isin(done["title"])].reset_index(drop=True)
    if todo.empty:
        print(f"All {len(table)} images are coded and saved in {out_csv}.")
        return
    state = {"i": 0}
    picture = widgets.Image(format="jpeg", width=420)
    buttons = {v: widgets.ToggleButtons(options=["?", "no", "yes"], value="?", description=v) for v in variables}
    save = widgets.Button(description="Save and next", button_style="success")
    status = widgets.HTML()

    def show():
        buffer = io.BytesIO()
        load_rgb(todo.loc[state["i"], "local_path"], 600).save(buffer, format="JPEG")
        picture.value = buffer.getvalue()
        for button in buttons.values():
            button.value = "?"
        status.value = f"<b>Image {len(done) + state['i'] + 1} of {len(table)}</b>"

    def on_save(_):
        if any(button.value == "?" for button in buttons.values()):
            status.value += " <span style='color:#c00'>Answer every question first.</span>"
            return
        record = {"title": todo.loc[state["i"], "title"], **{v: int(b.value == "yes") for v, b in buttons.items()}}
        pd.DataFrame([record]).to_csv(out_csv, mode="a", header=not out_csv.exists(), index=False)
        state["i"] += 1
        if state["i"] < len(todo):
            show()
        else:
            status.value = f"<b>Done!</b> {len(table)} images coded and saved in {out_csv}."
            save.disabled = True

    save.on_click(on_save)
    display(widgets.VBox([status, picture, *buttons.values(), save]))
    show()

code_images(coding_sample, "my_codes.csv")

When you have coded all 40 images, load your codes and look at how often you said "yes".

In [ ]:
codes = pd.read_csv("my_codes.csv")
print(f"{len(codes)} images coded")
codes[VARIABLES].mean().round(2).rename("share coded yes")

## 3. How reliable is the coding?

A code is only useful if it would come out the same when coded again, by you on another day or by
someone else with the same codebook. There are two kinds of reliability:

- **Stability** (intra-coder): does the same coder give the same code twice?
- **Reproducibility** (inter-coder): do different coders give the same code? This is the standard
  for publication.

We can test stability right away. Below are 20 of your 40 images again, in a new order. Code them
without looking back at your first answers. (For reproducibility, ask a colleague to code the
same 40 images with this notebook and compare their `my_codes.csv` with yours in exactly the same
way.)

In [ ]:
recode_sample = coding_sample.sample(20, random_state=7).reset_index(drop=True)
code_images(recode_sample, "my_recodes.csv")

Three common agreement measures:

- **Percent agreement**: the share of images where both codings match. Easy to read, but
  misleading for rare categories: if *police* is "no" in 95% of images, two coders who always say
  "no" agree 95% of the time without ever looking.
- **Cohen's kappa**: agreement corrected for the agreement you would expect by chance given how
  often each coder says "yes". 0 means chance level, 1 means perfect agreement.
- **Krippendorff's alpha**: a similar chance-corrected measure that works for any number of
  coders, for missing codes and for other levels of measurement. It is the most common standard
  in content analysis, where 0.8 is often treated as good and 0.667 as the lowest acceptable
  value for tentative conclusions.

> **✏️ Exercise 2**
>
> Compare your first coding with your recoding of the same 20 images.
>
> 1. For each variable, compute the share of "yes", percent agreement, Cohen's kappa and
>    Krippendorff's alpha (`krippendorff.alpha(reliability_data=[first, second],
>    level_of_measurement="nominal")`).
> 2. Which variable is least reliable? Is there a variable where percent agreement is high but
>    kappa or alpha is low, and why?
> 3. Show the images where your two codings of *protest* disagree. What made them hard?

In [ ]:
# Your code here

## 4. Labels that come for free

Every image already has a label we didn't have to code: the **category** it was collected from
(protest categories versus King's Day). Labels like these, from metadata, hashtags, search terms
or the account that posted an image, are tempting because they cost nothing and exist for every
image. But they describe *where an image was filed*, not *what it shows*.

> **✏️ Exercise 3**
>
> Compare the category label (`category_label` in `coding_sample`: 1 for the protest categories,
> 0 for King's Day) with your own *protest* code for the 40 images.
>
> 1. Make a cross-table and compute Cohen's kappa.
> 2. Show the images where the two disagree. What kinds of images are they?
> 3. What does this mean for using category labels to train or evaluate a classifier?

In [ ]:
# Your code here

## 5. A classic classifier on handcrafted features

Hand coding every image doesn't scale, so let's see how far the features from module 03 get us.
The plan:

1. compute the features for every image (the course helper contains the same feature code as
   module 03, without OCR);
2. **train** on all images *except* your 40 coded ones, using the free category labels, because
   that is the only label we have for every image;
3. estimate performance on the training images with **cross-validation**;
4. **test** the final model on your 40 coded images, which it has never seen.

Keeping the coded images out of training is essential: a model tested on images it was trained on
looks far better than it is.

In [ ]:
everything = pd.concat([protests.assign(category_label=1), koningsdag.assign(category_label=0)], ignore_index=True)
features = pd.DataFrame([course_data.image_features(p) for p in everything["local_path"]])
FEATURES = list(features.columns)
data = pd.concat([everything[["title", "local_path", "category_label"]], features], axis=1)

is_test = data["title"].isin(coding_sample["title"])
train, test = data[~is_test].reset_index(drop=True), data[is_test].reset_index(drop=True)
print(f"Training images: {len(train)} ({train['category_label'].mean():.0%} from protest categories)")
print(f"Test images (your coded sample): {len(test)}")
print("Features:", ", ".join(FEATURES))

The classes are unbalanced: most images come from the protest categories. A model that always
says "protest" would be right most of the time and useless. We therefore use
`class_weight="balanced"`, which makes mistakes on the smaller class count more during training,
and report **balanced accuracy** (the average of the accuracy within each class, so 0.5 is chance)
and the F1 score for each class.

We compare two models: a **logistic regression**, which adds up the features with one weight
each (after standardizing them), and a **random forest**, which combines many decision trees and
can capture thresholds and interactions ("bright *and* few faces").

In [ ]:
models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=SEED),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X, y = train[FEATURES], train["category_label"]

cv_predictions = {}
for name, model in models.items():
    pred = cross_val_predict(model, X, y, cv=cv)
    cv_predictions[name] = pred
    f1_neg, f1_pos = f1_score(y, pred, average=None)
    print(f"{name:<20} balanced accuracy {balanced_accuracy_score(y, pred):.2f}   "
          f"F1 King's Day {f1_neg:.2f}   F1 protest {f1_pos:.2f}")

ConfusionMatrixDisplay(confusion_matrix(y, cv_predictions["random forest"]),
                       display_labels=["King's Day", "protest"]).plot(colorbar=False)
plt.title("Random forest, cross-validated")
plt.show()

In [ ]:
forest = models["random forest"].fit(X, y)
importance = pd.Series(forest.feature_importances_, index=FEATURES).sort_values()
importance.plot.barh(figsize=(5, 3.5), title="Which features the forest relies on")
plt.xlabel("importance")
plt.show()

> **✏️ Exercise 4**
>
> Which **kind** of feature carries the signal? Group the features as below and, for each group on
> its own, compute the cross-validated balanced accuracy of the random forest. Then do the same for
> all features together. Does the best group make sense to you, given what distinguishes the two
> kinds of images?
>
> ```python
> GROUPS = {
>     "light": ["brightness", "contrast", "share_dark", "share_bright"],
>     "color": ["saturation", "colorfulness"],
>     "composition": ["edge_density", "bytes_per_pixel", "symmetry"],
>     "faces": ["n_faces", "largest_face_share"],
> }
> ```

In [ ]:
# Your code here

## 6. Testing against your codes

Now the real test: the 40 images you coded, which the model has never seen. We can score its
predictions against two things: the category label (what it was trained to predict) and your
*protest* code (what we actually care about).

In [ ]:
test = test.merge(codes[["title", "protest"]], on="title")
test["predicted"] = forest.predict(test[FEATURES])
test["prob_protest"] = forest.predict_proba(test[FEATURES])[:, 1]
test[["title", "category_label", "protest", "predicted", "prob_protest"]].head(10)

> **✏️ Exercise 5**
>
> 1. Compute the balanced accuracy of the predictions against the **category label** and against
>    **your protest code**. Which is higher, and why?
> 2. Show the images where the model's prediction disagrees with your code, with the predicted
>    probability of "protest" in the title. Look for a pattern in the mistakes.

In [ ]:
# Your code here

## 7. The semantic gap

Handcrafted features describe **how an image looks**: its light, colors, edges, and roughly how
many frontal faces it contains. Your codes describe **what it shows**: people protesting, signs
with messages, police. The distance between the two is known in computer vision as the
**semantic gap**. A classifier on low-level features can only cross it where the meaning happens
to correlate with the look (orange for King's Day), and those correlations break easily on new
data.

Closing the gap requires features that represent content: objects, people, text, scenes. Part 2
of the course does exactly that, first by letting a network learn its own features (module 05)
and then by borrowing features from networks trained on millions of images (module 06). Your 40
codes remain the test set: every later method can be scored against the same images.

> **✏️ Exercise 6**
>
> Pick two of the model's mistakes from exercise 6. For each, describe:
>
> 1. what the image shows (in terms of your codebook),
> 2. which features probably misled the model, and
> 3. what information a model would need to get it right.

*Your answer here*

## 8. Keep your codes, and start a scoreboard

Your 40 codes are the test set for the rest of the course, but Colab deletes files when a session
ends. The next cell does two things:

- it adds the random forest's result on your codes to a **scoreboard**, `my_results.csv`, which
  every later module adds to, so you can compare methods on exactly the same images;
- it copies `my_codes.csv`, `my_recodes.csv` and `my_results.csv` to a folder
  `pixels-frames-transformers` in your **Google Drive**. Colab asks for permission the first time.
  Later modules look for the files there.

If you would rather not connect Google Drive, download the three files from the file browser on
the left and upload them again at the start of later modules.

In [ ]:
course_data.log_score("random forest, handcrafted features", test["protest"], test["predicted"], module="04")
course_data.keep_my_files()
course_data.show_scores()

## Wrap-up

You have seen that:

- coding images starts with a unit of analysis and a codebook that says what must be visible;
- reliability can be checked within one coder (stability) and between coders (reproducibility),
  and chance-corrected measures like kappa and alpha matter most for rare categories;
- free labels from metadata describe where an image was filed, not what it shows;
- classifiers on handcrafted features can be trained and cross-validated like any other model,
  with balanced metrics when classes are unequal and a test set they have never seen;
- such classifiers learn what images look like, not what they mean: the semantic gap;
- your codes are worth keeping: they are the yardstick for every method that follows.

**Next:** module 05 opens up convolutional neural networks, which learn their own features from
pixels.